In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
propanolamine,75.11,3.86626,3.006102302,180.4056401,2,2
"""
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
propanolamine,H,propanolamine,e,2000.0,0.03
"""


model = PCSAFT(["propanolamine"], userlocations = [like_parameter, assoc_parameter])
print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2000.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_propanolamine.csv")
fix_line_endings("rhol_propanolamine.csv")

Fixed: satp_propanolamine.csv
Fixed: rhol_propanolamine.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

976.1844586341307
56.830969911280064


In [5]:
toestimate = [
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 5000.0,
        :guess   => 2400.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.0,
        :upper   => 1.0,
        :guess   => 0.4
    )
]

2-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 5000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 2400.0, :lower => 0.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.4, :lower => 0.0)

In [6]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_propanolamine.csv",
        "satp_propanolamine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 0.9155974955615135


In [7]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2269.2010464249497, 0.0903067168805119], PCSAFT{BasicIdeal, Float64}("propanolamine"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_propanolamine.csv",   my_saturation_p)

┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593



=== AAD: satp_propanolamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
360.8600    1333.508556   1349.036697   1.1645  
363.3694    1544.627360   1558.817503   0.9187  
365.8788    1784.469467   1796.965926   0.7003  
368.3882    2056.280729   2066.709309   0.5072  
370.8976    2363.594172   2371.566288   0.3373  
373.4069    2710.247477   2715.365199   0.1888  
375.9163    3100.400900   3102.262756   0.0601  
378.4257    3538.555602   3536.763221   0.0507  
380.9351    4029.572360   4023.737937   0.1448  
383.4445    4578.690622   4568.445186   0.2238  
385.9539    5191.547890   5176.550328   0.2889  
388.4633    5874.199371   5854.146172   0.3414  
390.9727    6633.137890   6607.773530   0.3824  
393.4820    7475.314007   7444.441900   0.4130  
395.9914    8408.156303   8371.650233   0.4342  
398.5008    9439.591816   9397.407718   0.4469  
401.0102    10578.066560  10530.254542  0.4520  
403.5196    11832.566116  11779.282568  0.4503  
406.0290    13212.63623

0.3019111312027907

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_propanolamine.csv",   my_liquid_density)


=== AAD: rhol_propanolamine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
293.1500    987.650000    987.456528    0.0196  
303.1500    979.650000    980.030579    0.0388  
313.1500    971.610000    972.589846    0.1008  
323.1500    963.520000    965.106741    0.1647  
333.1500    955.380000    957.555545    0.2277  
AARD = 0.1103%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.11033640641587113